In [2]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
sys.path.append(str(PROJECT_ROOT))

import pandas as pd

from src.config import (
    RAW_DATA_DIR,
    PROCESSED_DATA_DIR,
    REPORTS_DIR,
    RANDOM_SEED,
    TRAIN_FULL_FILE,
    TEST_FULL_FILE,
    PROMPT_SUBSET_FILE,
    ASSISTANT_SUBSET_FILE,
    ROBUSTNESS_SUBSET_FILE,
    SUMMARY_FILE,
)

from src.data_loader import load_yelp_dataset
from src.preprocess import (
    standardize_dataframe,
    add_text_statistics,
    get_quality_checks,
    get_class_distribution,
    get_length_summary,
    build_dataset_summary,
)
from src.sampling import stratified_sample

print("All imports loaded successfully.")
print("Project root:", PROJECT_ROOT)

All imports loaded successfully.
Project root: /home/rehan/Documents/yelp-ai-assignment


In [3]:
RAW_DATA_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DATA_DIR.mkdir(parents=True, exist_ok=True)
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

print("Folders are ready.")
print("Raw data dir      :", RAW_DATA_DIR)
print("Processed data dir:", PROCESSED_DATA_DIR)
print("Reports dir       :", REPORTS_DIR)

Folders are ready.
Raw data dir      : /home/rehan/Documents/yelp-ai-assignment/data/raw
Processed data dir: /home/rehan/Documents/yelp-ai-assignment/data/processed
Reports dir       : /home/rehan/Documents/yelp-ai-assignment/outputs/reports


In [4]:
train_df, test_df = load_yelp_dataset()

print("Dataset loaded successfully.")
print("Train shape:", train_df.shape)
print("Test shape :", test_df.shape)
train_df.head()

Dataset loaded successfully.
Train shape: (650000, 2)
Test shape : (50000, 2)


,label,text
0,4,dr. goldberg offers everything i look for in a...
1,1,"Unfortunately, the frustration of being Dr. Go..."
2,3,Been going to Dr. Goldberg for over 10 years. ...
3,3,Got a letter in the mail last week that said D...
4,0,I don't know what Dr. Goldberg was like before...


In [5]:
train_df = standardize_dataframe(train_df)
test_df = standardize_dataframe(test_df)

print("Standardized train columns:", train_df.columns.tolist())
print("Standardized test columns :", test_df.columns.tolist())

train_df.head()

Standardized train columns: ['text', 'label_raw', 'stars']
Standardized test columns : ['text', 'label_raw', 'stars']


,text,label_raw,stars
0,dr. goldberg offers everything i look for in a...,4,5
1,"Unfortunately, the frustration of being Dr. Go...",1,2
2,Been going to Dr. Goldberg for over 10 years. ...,3,4
3,Got a letter in the mail last week that said D...,3,4
4,I don't know what Dr. Goldberg was like before...,0,1


In [6]:
train_df = add_text_statistics(train_df)
test_df = add_text_statistics(test_df)

train_df.head()

,text,label_raw,stars,char_length,word_length
0,dr. goldberg offers everything i look for in a...,4,5,534,93
1,"Unfortunately, the frustration of being Dr. Go...",1,2,643,115
2,Been going to Dr. Goldberg for over 10 years. ...,3,4,495,97
3,Got a letter in the mail last week that said D...,3,4,261,53
4,I don't know what Dr. Goldberg was like before...,0,1,1143,211


In [7]:
train_checks = get_quality_checks(train_df)
test_checks = get_quality_checks(test_df)

print("TRAIN CHECKS")
print(train_checks)

print("\nTEST CHECKS")
print(test_checks)

TRAIN CHECKS
{'rows': 650000, 'null_text': 0, 'null_label_raw': 0, 'null_stars': 0, 'empty_reviews': 0, 'duplicate_reviews': 0}

TEST CHECKS
{'rows': 50000, 'null_text': 0, 'null_label_raw': 0, 'null_stars': 0, 'empty_reviews': 0, 'duplicate_reviews': 0}


In [8]:
train_df = standardize_dataframe(train_df)
test_df = standardize_dataframe(test_df)

In [9]:
train_df, test_df = load_yelp_dataset()

In [10]:
train_df = standardize_dataframe(train_df)
test_df = standardize_dataframe(test_df)

In [11]:
train_df = add_text_statistics(train_df)
test_df = add_text_statistics(test_df)

In [12]:
train_checks = get_quality_checks(train_df)
test_checks = get_quality_checks(test_df)

print("TRAIN CHECKS")
print(train_checks)

print("\nTEST CHECKS")
print(test_checks)

TRAIN CHECKS
{'rows': 650000, 'null_text': 0, 'null_label_raw': 0, 'null_stars': 0, 'empty_reviews': 0, 'duplicate_reviews': 0}

TEST CHECKS
{'rows': 50000, 'null_text': 0, 'null_label_raw': 0, 'null_stars': 0, 'empty_reviews': 0, 'duplicate_reviews': 0}


In [13]:
print("TRAIN CLASS DISTRIBUTION")
print(get_class_distribution(train_df))

print("\nTEST CLASS DISTRIBUTION")
print(get_class_distribution(test_df))

TRAIN CLASS DISTRIBUTION
{1: 130000, 2: 130000, 3: 130000, 4: 130000, 5: 130000}

TEST CLASS DISTRIBUTION
{1: 10000, 2: 10000, 3: 10000, 4: 10000, 5: 10000}


In [ ]:
prompt_subset = stratified_sample(test_df, label_col="stars", n_total=500, random_state=RANDOM_SEED)
assistant_subset = stratified_sample(test_df, label_col="stars", n_total=100, random_state=RANDOM_SEED + 1)
robustness_subset = stratified_sample(test_df, label_col="stars", n_total=200, random_state=RANDOM_SEED + 2)

In [15]:
train_df.to_csv(TRAIN_FULL_FILE, index=False)
test_df.to_csv(TEST_FULL_FILE, index=False)
prompt_subset.to_csv(PROMPT_SUBSET_FILE, index=False)
assistant_subset.to_csv(ASSISTANT_SUBSET_FILE, index=False)
robustness_subset.to_csv(ROBUSTNESS_SUBSET_FILE, index=False)

In [16]:
print("TRAIN CLASS DISTRIBUTION")
print(get_class_distribution(train_df))

print("\nTEST CLASS DISTRIBUTION")
print(get_class_distribution(test_df))

TRAIN CLASS DISTRIBUTION
{1: 130000, 2: 130000, 3: 130000, 4: 130000, 5: 130000}

TEST CLASS DISTRIBUTION
{1: 10000, 2: 10000, 3: 10000, 4: 10000, 5: 10000}


In [17]:
print("TRAIN LENGTH SUMMARY")
print(get_length_summary(train_df))

print("\nTEST LENGTH SUMMARY")
print(get_length_summary(test_df))

TRAIN LENGTH SUMMARY
{'avg_char_length': np.float64(732.33), 'avg_word_length': np.float64(134.1), 'min_word_length': 1, 'max_word_length': 1052}

TEST LENGTH SUMMARY
{'avg_char_length': np.float64(733.42), 'avg_word_length': np.float64(134.29), 'min_word_length': 1, 'max_word_length': 1009}


In [18]:
prompt_subset = stratified_sample(
    test_df,
    label_col="stars",
    n_total=500,
    random_state=RANDOM_SEED
)

assistant_subset = stratified_sample(
    test_df,
    label_col="stars",
    n_total=100,
    random_state=RANDOM_SEED + 1
)

robustness_subset = stratified_sample(
    test_df,
    label_col="stars",
    n_total=200,
    random_state=RANDOM_SEED + 2
)

print("Prompt subset shape    :", prompt_subset.shape)
print("Assistant subset shape :", assistant_subset.shape)
print("Robustness subset shape:", robustness_subset.shape)

Prompt subset shape    : (500, 5)
Assistant subset shape : (100, 5)
Robustness subset shape: (200, 5)


In [19]:
print("PROMPT SUBSET DISTRIBUTION")
print(prompt_subset["stars"].value_counts().sort_index())

print("\nASSISTANT SUBSET DISTRIBUTION")
print(assistant_subset["stars"].value_counts().sort_index())

print("\nROBUSTNESS SUBSET DISTRIBUTION")
print(robustness_subset["stars"].value_counts().sort_index())

PROMPT SUBSET DISTRIBUTION
stars
1    100
2    100
3    100
4    100
5    100
Name: count, dtype: int64

ASSISTANT SUBSET DISTRIBUTION
stars
1    20
2    20
3    20
4    20
5    20
Name: count, dtype: int64

ROBUSTNESS SUBSET DISTRIBUTION
stars
1    40
2    40
3    40
4    40
5    40
Name: count, dtype: int64


In [20]:
train_df.to_csv(TRAIN_FULL_FILE, index=False)
test_df.to_csv(TEST_FULL_FILE, index=False)
prompt_subset.to_csv(PROMPT_SUBSET_FILE, index=False)
assistant_subset.to_csv(ASSISTANT_SUBSET_FILE, index=False)
robustness_subset.to_csv(ROBUSTNESS_SUBSET_FILE, index=False)

print("Saved files:")
print(TRAIN_FULL_FILE)
print(TEST_FULL_FILE)
print(PROMPT_SUBSET_FILE)
print(ASSISTANT_SUBSET_FILE)
print(ROBUSTNESS_SUBSET_FILE)

Saved files:
/home/rehan/Documents/yelp-ai-assignment/data/processed/yelp_train_full.csv
/home/rehan/Documents/yelp-ai-assignment/data/processed/yelp_test_full.csv
/home/rehan/Documents/yelp-ai-assignment/data/processed/yelp_prompt_subset_500.csv
/home/rehan/Documents/yelp-ai-assignment/data/processed/yelp_assistant_subset_100.csv
/home/rehan/Documents/yelp-ai-assignment/data/processed/yelp_robustness_subset_200.csv


In [21]:
summary_text = build_dataset_summary(train_df, test_df)

with open(SUMMARY_FILE, "w", encoding="utf-8") as f:
    f.write(summary_text)

print("Dataset summary saved to:")
print(SUMMARY_FILE)

Dataset summary saved to:
/home/rehan/Documents/yelp-ai-assignment/outputs/reports/dataset_summary.txt


In [22]:
print(summary_text[:1500])

YELP REVIEW FULL - DATASET SUMMARY

TRAIN DATASET
-------------
Rows: 650000
Null text: 0
Null label_raw: 0
Null stars: 0
Empty reviews: 0
Duplicate reviews: 0
Class distribution: {1: 130000, 2: 130000, 3: 130000, 4: 130000, 5: 130000}
Length summary: {'avg_char_length': np.float64(732.33), 'avg_word_length': np.float64(134.1), 'min_word_length': 1, 'max_word_length': 1052}

TEST DATASET
------------
Rows: 50000
Null text: 0
Null label_raw: 0
Null stars: 0
Empty reviews: 0
Duplicate reviews: 0
Class distribution: {1: 10000, 2: 10000, 3: 10000, 4: 10000, 5: 10000}
Length summary: {'avg_char_length': np.float64(733.42), 'avg_word_length': np.float64(134.29), 'min_word_length': 1, 'max_word_length': 1009}
